### ***Numerical_list and Categorical_list***

In [ ]:
import pandas as pd

numerical_list = []
categorical_list = []
df = pd.DataFrame()

for i in df.columns:
    if df[i].dtype == 'O': # dtype이 O오브젝트라고 명시되어있음.
        categorical_list.append(i)
    else:
        numerical_list.append(i)

print("Numerical_list:", numerical_list)
print("Categorical_list:", categorical_list)


Numerical_list: ['UDI', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
Categorical_list: ['Product ID', 'Type']


In [ ]:
list_categorical_columns = list(df.select_dtypes(include=['object']).columns)
list_numeric_columns = list(df.select_dtypes(include=['float64','int64']).columns)

### Categorical Variable + Staitstics Significance testing

카이제곱 검정 (chisqaure-test)
카이제곱 검정은 두 범주형(Categorical) 변수에 대한 분석 방법.
예를 들어, 성별에 따른 신용카드 발급 유무.
본 실습에서는 독립 변수중의 categorical column data와 y(target_status) 간의 관계를 검증.
pvalue는 0.05 이하인 경우, 귀무가설을 기각한다고 정한다. 귀무가설은 X와 Y는 서로 관계가 없다는 것.

#### 카이제곱 검정 3가지 유형:

1. 적합도 검정 (Goodness of fit)
- 변수 1개
- 기존에 알려준 기준이 존재하는 검정
- 예시) 상자 안에 공 3개가 같은 비율로 알려져 있음. 공 100개를 뽑았을 때, 각 색의 비율이 구해짐. -> 기존에 알려진 공 비율 분포를 따르는지 검정
- 귀무가설 : 변수 X의 관측분포와 기대 분포가 동일
- 대립가설 : 변수 X의 관측분포와 기대분포가 동일

2.   독립성 검정 (Test of independence)
- 변수 2개
- 범주형 두 변수가 서로 연관되어 있는지 여부를 검정
- 예시) 성별과 흡연 여부 관계를 알고 싶어서 200명을 추출하여 조사한 경우.
- 귀무가설 : 변수 X와 Y는 서로 독립
- 대립가설 : 변수 X와 Y는 서로 독립이 아님

3. 동질성 검정 (Test of Homogeneity) 
- 변수 2개
- 범주형 두 변수의 관계를 알기 위한 검정은 아님
- 각 그룹들이 동질한지 알고 싶은 검정
- 예시) 남자와 여자 흡연율 차이가 있는지 흡연율을 조사한 후, 두 그룹의 흡연율이 같은지 여부를 검정
- 귀무가설 : 각 그룹의 확률분포가 동일
- 대립가설 : 각 그룹의 확률분포가 동일하지 않음

In [ ]:
list_meaningful_column_by_chi = [] # 비어있는 리스트를 만들어서 카이제곱 검정을 통과한 column들을 넣어주려고한다. 
for column_name in list_categorical_columns: # 카테고리컬 컬럼들을 하나씩 돌면서
  statistic, pvalue, _, _ = chi2_contingency(pd.crosstab(df[target_column], df[column_name])) # 카이제곱 검정을 진행한다. 
  if pvalue <= 0.05: # pvalue가 0.05 이하인 경우에만
    list_meaningful_column_by_chi.append(column_name) # 리스트에 추가한다.
  print(column_name, ", ",statistic,", ", pvalue) # 각 column에 대한 카이제곱 통계량과 pvalue를 출력한다.
print("all categorical columns : ", len(list_categorical_columns)) # 모든 카테고리컬 컬럼의 개수를 출력한다.
print("selected columns by chi : ", len(list_meaningful_column_by_chi), list_meaningful_column_by_chi) # 카이제곱 검정을 통과한 컬럼의 개수와 리스트를 출력한다.

## Check Skewness and Kurtosis

In [ ]:
for column_name in list_numeric_columns: # 수치형 컬럼들을 하나씩 돌면서
  print(column_name, "skew : ", skew(df[column_name]), "kur : ", kurtosis(df[column_name]) ) # 각 column에 대한 왜도와 첨도를 출력한다.
  
# 추후 scaling을 활용한 feature preprocessing의 필요성 확인

##### VIF Analysis
-   일반적으로는는 10이상인인 경우 다중공선성이이 있다고 가정
-   high correlation 컬럼을 제거하기 전에 한 번 더 검정을 진행 (doublecheck, 개인의 판단에 따라서 진행하지 않아도됨)

In [ ]:
# calculate_vif function
#### to do ####
import variance_inflation_factor

def calculate_vif(df_target):
  vif = pd.DataFrame()
  vif["VIF_Factor"] = [variance_inflation_factor(df_target.values, i) for i in range(df_target.shape[1])]
  vif["Feature"] = df_target.columns
  return vif

df_vif = df[list_numeric_columns].copy()

def calculate_cif(df_target):
    vif = pd.DataFrame() # 빈 데이터프레임 생성
    vif["VIF_Factor"] = [variance_inflation_factor(df_target.values, i) for i in range(df_target.shape[1])] # VIF_Factor 열 생성. 여기서 df_target.shape[1]은 df_target의 열 개수를 의미한다.
    vif["feature"] = df_target.columns
    return vif

calculate_vif(df[list_numeric_columns]) # VIF 계산 함수 호출

##### 일원분산분석 (ANOVA)
*   카테고리별 numeric data 분포 차이를 검증
*   전제 : 정규성, 등분산성, 독립성 (만족하지 않으면 해당 검정을 신뢰할 수 없음)
*   본 실습에서는 target_status 에 따른 numeric column data 분포 관계를 검증.

##### 정규성 검정
*   귀무가설 : 모집단의 분포는 정규 분포이다
*   검정 방법 : Shpiro-Wilks Test, qqplot